# Notebook 46 — Option 5: Gate C — honest reduced-posterior classification report

## Purpose

One of three notebooks (44/45/46) built 2026-09-06. **You are running this notebook,
not Claude** — read the markdown cells, they explain what to watch for.

## The idea

`docs/Manuscript/system_ii_sbc_repair_directions.md` and
`system_ii_sbi_solidification_plan.md` both pre-register a fallback ("Gate C"): if
nothing fully calibrates, report a **reduced posterior with explicitly documented
limits** rather than chasing further fixes indefinitely. This notebook turns that from a
prose caveat into an actual, quantified artifact:

1. A **per-parameter calibration status table**, sourced directly from the existing SBC
   seed-sweep results (not re-derived) — which parameters are genuinely calibrated, which
   fail, and *why* (e.g. `alpha`'s failure is concentrated at its prior's untested upper
   boundary, a region no named Wu2003 fault scenario ever needs — see `HANDOFF.md`).
2. A **confidence/abstention layer** on top of the existing best posterior
   (`mixed_alpha_uniform_prior`, seed 0): flag each classification call as `confident` or
   `boundary-adjacent` based on how much posterior mass falls in the region SBC could not
   validate. This gives three honest numbers instead of one optimistic one: the naive
   macro-F1 (as reported elsewhere), the macro-F1 restricted to confident calls only, and
   the abstention rate.

This does not try to improve the underlying posterior — it uses the best one already
trained and validated this session, and makes its limitations an explicit, reportable
part of the classification output instead of leaving them as an unstated caveat.


In [ ]:
import pathlib, pickle, sys

import numpy as np
import pandas as pd
import torch

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from cstr_sbi.recycle.physics import PARAM_NAMES
from cstr_sbi.recycle.scenarios import list_closed_loop, FAULT_UNITS
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries

RESULTS = ROOT / "results"
SBI_LOGS = ROOT / "sbi-logs"

STRUCTURE = "S-C.2"
NOISE_PCT = 0.003
N_REPS, N_POST = 30, 200
THRESH, Z0_NOM = 0.85, 0.90
SEED = 0

# The best-already-validated posterior this session: alpha-only mixed prior, seed 0.
# ("genuinely fixed, not shrunk" for alpha away from the prior's upper boundary --
# HANDOFF.md 2026-09-05.)
POSTERIOR_PATH = SBI_LOGS / "wu2003_posterior_sc2_true_n1000_zscorenone_seed0_mixedprior_stage1bfix.pkl"
SBC_SWEEP_CSV = RESULTS / "sc2_mixedprior_zscorenone_seedsweep_n10_mixedprior_stage1bfix.csv"

# alpha's boundary: prior is Uniform(0.40, 1.20); direct posterior inspection this
# session found the fix genuinely holds for alpha roughly in [0.40, 1.11] and becomes
# untested/boundary-adjacent above that. This threshold is the one free parameter in
# this notebook -- adjust if the underlying posterior/prior bounds change.
ALPHA_VALIDATED_UPPER = 1.11


## Step 1 — per-parameter calibration status table

Loaded directly from the existing seed-sweep CSV (not re-derived), seed 0 (the posterior
being used for classification below).


In [ ]:
sbc_df = pd.read_csv(SBC_SWEEP_CSV)
seed0 = sbc_df[sbc_df["seed"] == SEED].iloc[0]

rows = []
for name in PARAM_NAMES:
    reduced_p = seed0[f"{name}_reduced_ks"]
    confirm_p = seed0[f"{name}_confirm_ks"]
    status = "PASS" if confirm_p > 0.05 else "FAIL"
    note = ""
    if name == "alpha":
        note = (f"fails formal SBC due to the prior's untested upper boundary "
                 f"(>{ALPHA_VALIDATED_UPPER}); direct posterior inspection confirmed "
                 f"genuinely fixed below that -- see HANDOFF.md 2026-09-05.")
    rows.append({"parameter": name, "reduced_N120_ks_p": reduced_p, "confirm_N400_ks_p": confirm_p,
                  "status": status, "note": note})

calib_table = pd.DataFrame(rows)
calib_table.to_csv(RESULTS / "46_gate_c_calibration_status.csv", index=False)
print(calib_table.to_string(index=False))


## Step 2 — classification with a confidence/abstention layer

`classify_with_confidence()` computes the usual posterior-mass-threshold class
prediction, plus the fraction of posterior mass for `alpha` that falls above
`ALPHA_VALIDATED_UPPER` (the untested/boundary-adjacent region). A replicate is flagged
`boundary_adjacent` if that fraction exceeds `BOUNDARY_MASS_THRESHOLD`.


In [ ]:
BOUNDARY_MASS_THRESHOLD = 0.20  # flag if >20% of posterior alpha-mass sits in the untested region

def sample_fault_unit(theta_row, thresh=THRESH, z0_nom=Z0_NOM):
    alpha, beta_r, eta_col, xi_reb, z_A0 = theta_row
    reactor_bad = (alpha < thresh) or (beta_r < thresh)
    column_bad = (eta_col < thresh) or (xi_reb < thresh)
    feed_bad = z_A0 < thresh * z0_nom
    n_bad = int(reactor_bad) + int(column_bad) + int(feed_bad)
    if n_bad == 0:
        return "healthy"
    if n_bad >= 2:
        return "multi"
    if reactor_bad:
        return "reactor"
    if column_bad:
        return "column"
    return "feed"


def classify_with_confidence(samples):
    labels = [sample_fault_unit(row) for row in samples]
    probs = {u: float(np.mean([l == u for l in labels])) for u in FAULT_UNITS}
    pred = max(probs, key=probs.__getitem__)
    boundary_mass = float(np.mean(samples[:, 0] > ALPHA_VALIDATED_UPPER))  # alpha is index 0
    confidence = "boundary_adjacent" if boundary_mass > BOUNDARY_MASS_THRESHOLD else "confident"
    return pred, probs, boundary_mass, confidence


def compute_classification_metrics(preds, trues, classes=FAULT_UNITS):
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1_per_class = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_per_class[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    macro_f1 = float(np.mean(list(f1_per_class.values())))
    accuracy = float(np.diag(cm).sum() / cm.sum()) if cm.sum() > 0 else 0.0
    return {"confusion_matrix": cm, "class_names": classes, "per_class_f1": f1_per_class,
            "macro_f1": macro_f1, "accuracy": accuracy}


with open(POSTERIOR_PATH, "rb") as f:
    payload = pickle.load(f)
posterior = payload["posterior"]
print(f"Loaded {POSTERIOR_PATH.name} (structure={payload.get('structure')}, seed={payload.get('torch_seed')})")

torch.manual_seed(20260705)
y0_sb = nominal_warm_start("S-B")
rows = []
for sc in list_closed_loop():
    theta_np = np.asarray(sc.theta(), dtype=np.float32)
    true_class = sc.fault_unit()
    t_h, raw_obs = deterministic_window(sc, structure="S-B", y0=y0_sb)
    t_h_arr = np.asarray(t_h)
    rng = np.random.default_rng(20260705)
    reps = noisy_replicates(raw_obs, n_replicates=N_REPS, rng=rng, noise_pct=NOISE_PCT)
    for i in range(N_REPS):
        s = compute_summaries(reps[i], STRUCTURE, t_h_arr, theta=theta_np, rng=rng)
        with torch.no_grad():
            samp = posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
                                     show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        pred, probs, boundary_mass, confidence = classify_with_confidence(samp)
        rows.append({"scenario": sc.name, "replicate": i, "true_class": true_class, "pred_class": pred,
                      "boundary_mass": boundary_mass, "confidence": confidence, **probs})
    print(f"  {sc.name:22s} true={true_class}")

gate_c_df = pd.DataFrame(rows)
gate_c_df.to_csv(RESULTS / "46_gate_c_classification_with_confidence.csv", index=False)


## Step 3 — three-tier metrics report

The number that matters for an honest Gate-C manuscript claim is the **confident-subset
macro-F1** alongside the **abstention rate** — not the naive full-sample number alone.


In [ ]:
naive_metrics = compute_classification_metrics(gate_c_df["pred_class"].tolist(), gate_c_df["true_class"].tolist())
confident_df = gate_c_df[gate_c_df["confidence"] == "confident"]
abstention_rate = 1.0 - len(confident_df) / len(gate_c_df)
if len(confident_df) > 0:
    confident_metrics = compute_classification_metrics(confident_df["pred_class"].tolist(), confident_df["true_class"].tolist())
else:
    confident_metrics = None

print("=" * 70)
print(f"(a) NAIVE full-sample macro-F1:      {naive_metrics['macro_f1']:.3f}  (accuracy={naive_metrics['accuracy']:.3f})")
if confident_metrics is not None:
    print(f"(b) CONFIDENT-subset macro-F1:        {confident_metrics['macro_f1']:.3f}  (accuracy={confident_metrics['accuracy']:.3f})")
print(f"    abstention rate (boundary-adjacent): {abstention_rate:.1%}  ({len(gate_c_df) - len(confident_df)}/{len(gate_c_df)} replicates)")
print("=" * 70)

print("\n(c) Per-scenario breakdown -- which scenarios are solid vs. flagged:")
per_scenario = gate_c_df.groupby("scenario").agg(
    true_class=("true_class", "first"),
    pct_confident=("confidence", lambda s: (s == "confident").mean()),
    majority_pred=("pred_class", lambda s: s.value_counts().idxmax()),
).reset_index()
print(per_scenario.to_string(index=False))
per_scenario.to_csv(RESULTS / "46_gate_c_per_scenario_summary.csv", index=False)


## Interpretation notes

- If the **confident-subset macro-F1** is meaningfully higher than the naive one, that's
  the actual, defensible Gate-C number to report: "the posterior is reliable for X% of
  cases (macro-F1=Y), and honestly flags the rest as needing independent confirmation"
  — a legitimate scientific claim, unlike silently reporting the naive (lower, noisier)
  number as if it were uniform in reliability.
- If confident-subset performance is *not* meaningfully better than naive, that would
  mean the boundary-adjacency flag isn't actually capturing the dominant failure mode —
  worth checking whether `alpha`'s failures are concentrated elsewhere (e.g. check
  `finding_alpha_beta_r_banana`/`finding_sa_headline_retracted` memories for other
  candidate confounds), or whether `beta_r`/`eta_col`/`xi_reb` also need a similar
  confidence flag (this notebook only flags `alpha`).
- The recommended manuscript language for Gate C, if this pans out: report the reduced/
  confident-subset number as the headline, state the per-parameter calibration table
  (Step 1) plainly, and describe the abstention mechanism as a designed safety feature
  of the deployed system rather than as a limitation to hide.


## Final acceptance criteria

| # | Criterion | Threshold | Rationale |
|---|---|---|---|
| 1 | Confident-subset macro-F1 exceeds naive macro-F1 | strictly greater | the confidence flag should actually identify more-reliable calls, not just relabel noise |
| 2 | Abstention rate is reasonable | 0% < rate ≤ 50% | 0% means the flag never fires (not doing anything); > 50% means the safety mechanism makes the system impractical |
| 3 | Confident-subset macro-F1 usability bar | ≥ 0.15 | a modest but real improvement over the naive alpha-only-mixed baseline (0.11) |


In [ ]:
HAS_CONFIDENT = confident_metrics is not None
naive_f1 = naive_metrics["macro_f1"]
confident_f1 = confident_metrics["macro_f1"] if HAS_CONFIDENT else float("nan")

criteria = [
    {"criterion": "1. Confident-subset macro-F1 > naive macro-F1", "value": confident_f1 - naive_f1 if HAS_CONFIDENT else float("nan"),
     "threshold": 0.0, "pass": (HAS_CONFIDENT and confident_f1 > naive_f1)},
    {"criterion": "2. Abstention rate in (0%, 50%]", "value": abstention_rate,
     "threshold": 0.50, "pass": (0.0 < abstention_rate <= 0.50)},
    {"criterion": "3. Confident-subset macro-F1 >= 0.15", "value": confident_f1,
     "threshold": 0.15, "pass": (HAS_CONFIDENT and confident_f1 >= 0.15)},
]

print("=" * 78)
print(f"{'Criterion':<45s} {'Value':>10s} {'Threshold':>10s} {'Result':>8s}")
print("-" * 78)
for c in criteria:
    val_str = f"{c['value']:.3f}" if c["value"] == c["value"] else "N/A"
    thr_str = f"{c['threshold']:.2f}" if isinstance(c["threshold"], float) else str(c["threshold"])
    print(f"{c['criterion']:<45s} {val_str:>10s} {thr_str:>10s} {'PASS' if c['pass'] else 'FAIL':>8s}")
print("=" * 78)

if not HAS_CONFIDENT:
    print("\nVERDICT: INVALID -- zero replicates were flagged confident; the confidence "
          "mechanism as configured abstains on everything. Reconsider ALPHA_VALIDATED_UPPER "
          "or BOUNDARY_MASS_THRESHOLD.")
elif all(c["pass"] for c in criteria):
    print("\nVERDICT: PASS -- this is a legitimate, defensible Gate-C manuscript claim: "
          "report the confident-subset macro-F1 as the headline number, state the "
          "abstention rate plainly, and describe the mechanism as a designed safety "
          "feature rather than a limitation to hide.")
elif criteria[0]["pass"]:
    print("\nVERDICT: PARTIAL -- the confidence flag helps directionally, but either the "
          "abstention rate or the absolute confident-subset performance isn't yet in a "
          "clearly reportable range. See the per-scenario breakdown (Step 3) for detail.")
else:
    print("\nVERDICT: FAIL -- the boundary-adjacency flag isn't capturing the dominant "
          "failure mode. Consider whether alpha's failures are concentrated elsewhere, or "
          "whether beta_r/eta_col/xi_reb also need a similar confidence flag (this "
          "notebook only flags alpha).")
